In [2]:
import pandas as pd
import numpy as np

In [3]:
df= pd.read_csv("master_dataset.csv")
df.head(5)

,raceId,year,round,circuitId,race_name,circuit_name,location,country,driverStandingsId,driverId,...,constructor_final_position,wins_constructor,qualifying_position,q1,q2,q3,pit_stop_count,pit_duration,avg_lap_time,total_laps
0,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68609,20,...,1,1,3.0,83.348,81.944,81.838,1.0,21.787,92.642810,58.0
1,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68610,1,...,1,1,1.0,82.824,82.051,81.164,1.0,21.821,92.729638,58.0
2,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68611,8,...,1,1,2.0,83.096,82.507,81.828,1.0,21.421,92.751586,58.0
3,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68612,817,...,1,1,5.0,83.494,82.897,82.152,1.0,21.440,92.764690,58.0
4,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68613,4,...,1,1,11.0,83.597,83.692,NaN,1.0,22.573,93.123603,58.0


In [4]:
# Driver form (average finishing position/starting position) - for the previous 5
#   SORT
df= df.sort_values(by=["driverId","year", 'round'], ascending=True)
#   POSITION RATIO 
# position ratio = final_position/qualifying position
df['pos_ratio'] = df["final_position"] / df['qualifying_position']
# grouping the drivers
driver_groups = df.groupby('driverId')['pos_ratio']
# calculating the form of the driver
df['driver_form'] = driver_groups.apply(
    lambda x : x.shift(1).rolling(window=5, min_periods=1).mean()
).reset_index(level=0, drop= True)

df['driver_form'] = df['driver_form'].fillna(1.0)


# verification
hamilton_form = df[
    df['driverId'] == 1
][['year', 'round', 'pos_ratio', 'driver_form']]

print(hamilton_form.tail(10))

      year  round  pos_ratio  driver_form
2876  2024     15   0.545455     2.193333
2903  2024     16   1.000000     1.769091
2919  2024     17   0.857143     1.649091
2941  2024     18   2.000000     1.020519
2963  2024     19   0.315789     1.180519
2986  2024     20   1.000000     0.943677
3017  2024     21   0.437500     1.034586
3042  2024     22   0.700000     0.922086
3055  2024     23   1.166667     0.890658
3078  2024     24   0.388889     0.723991


The driver form feature generated takes a look at the driver form, i.e his performance for the recent 5 races. The logic impremented is .rolling(window=5), this takes only the previous 5 races, takes the average of the position ratio and displays it as the driver form. Position ratio is calculated as (final position / qualifying position), 1 means the player finished where he started, less than 1 means he finished better than he started with, this logic continues to the drivr form and the closer the form is to 0, the better the form of the driver but only for the previous 5 races. For modelling purpose the last round of the last year (2024) depicts his average performace in the last 5 races

In [5]:
df.head(5)

,raceId,year,round,circuitId,race_name,circuit_name,location,country,driverStandingsId,driverId,...,qualifying_position,q1,q2,q3,pit_stop_count,pit_duration,avg_lap_time,total_laps,pos_ratio,driver_form
1,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68610,1,...,1.0,82.824,82.051,81.164,1.0,21.821,92.729638,58.0,2.0,1.000
39,990,2018,2,3,Bahrain Grand Prix,Bahrain International Circuit,Sakhir,Bahrain,68690,1,...,4.0,89.396,88.458,88.220,1.0,24.302,96.990386,57.0,0.5,2.000
41,991,2018,3,17,Chinese Grand Prix,Shanghai International Circuit,Shanghai,China,68670,1,...,4.0,93.283,91.914,91.675,1.0,22.464,102.738661,56.0,0.5,1.250
61,992,2018,4,73,Azerbaijan Grand Prix,Baku City Circuit,Baku,Azerbaijan,68710,1,...,2.0,102.693,102.676,101.677,1.0,20.377,122.044922,51.0,0.5,1.000
81,993,2018,5,4,Spanish Grand Prix,Circuit de Barcelona-Catalunya,Montmeló,Spain,68730,1,...,1.0,77.633,77.166,76.173,1.0,22.085,86.817758,66.0,1.0,0.875


In [6]:
# Group by driver AND circuit
circuit_groups= df.groupby(['driverId', 'circuitId'])['final_position']
df['driver_circuit_performance'] = circuit_groups.transform(
    lambda x: x.shift(1).expanding( min_periods=1).mean()
)
df['driver_circuit_performance'] = df['driver_circuit_performance'].fillna(10.0)

# verification
hamilton_silverstone = df[
    (df['driverId'] == 1) & 
    (df['circuitId'] == 9)  # circuitId 9 is Silverstone
].sort_values(by='year', ascending=True)
print(hamilton_silverstone[['year', 'final_position', 'driver_circuit_performance']])

      year  final_position  driver_circuit_performance
181   2018               2                   10.000000
601   2019               1                    2.000000
916   2020               1                    1.500000
924   2020               1                    1.333333
1419  2021               2                    1.250000
1837  2022               6                    1.400000
2299  2023               4                    2.166667
2813  2024               8                    2.428571


This feature "driver_circuit_performance" takes the performance metric of "finishing position" for a "driver-circuit" group (e.g hamilton_silverstone). This also works on the logic as before and looks as the average of the finishing position for all the races, telling us the historic performance of the driver in that circuit. The lower the number i.e closer to 1 the better the form of the driver.

In [7]:
df['driver_circuit_performance_5'] = circuit_groups.transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)
df['driver_circuit_performance_5'] = df['driver_circuit_performance_5'].fillna(10.0)

df.head(10)

,raceId,year,round,circuitId,race_name,circuit_name,location,country,driverStandingsId,driverId,...,q2,q3,pit_stop_count,pit_duration,avg_lap_time,total_laps,pos_ratio,driver_form,driver_circuit_performance,driver_circuit_performance_5
1,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68610,1,...,82.051,81.164,1.0,21.821,92.729638,58.0,2.000000,1.000000,10.0,10.0
39,990,2018,2,3,Bahrain Grand Prix,Bahrain International Circuit,Sakhir,Bahrain,68690,1,...,88.458,88.220,1.0,24.302,96.990386,57.0,0.500000,2.000000,10.0,10.0
41,991,2018,3,17,Chinese Grand Prix,Shanghai International Circuit,Shanghai,China,68670,1,...,91.914,91.675,1.0,22.464,102.738661,56.0,0.500000,1.250000,10.0,10.0
61,992,2018,4,73,Azerbaijan Grand Prix,Baku City Circuit,Baku,Azerbaijan,68710,1,...,102.676,101.677,1.0,20.377,122.044922,51.0,0.500000,1.000000,10.0,10.0
81,993,2018,5,4,Spanish Grand Prix,Circuit de Barcelona-Catalunya,Montmeló,Spain,68730,1,...,77.166,76.173,1.0,22.085,86.817758,66.0,1.000000,0.875000,10.0,10.0
101,994,2018,6,6,Monaco Grand Prix,Circuit de Monaco,Monte-Carlo,Monaco,68750,1,...,71.584,71.232,1.0,23.739,79.382308,78.0,0.333333,0.900000,10.0,10.0
121,995,2018,7,7,Canadian Grand Prix,Circuit Gilles Villeneuve,Montreal,Canada,68770,1,...,71.740,70.996,1.0,23.335,78.425529,68.0,0.500000,0.566667,10.0,10.0
141,996,2018,8,34,French Grand Prix,Circuit Paul Ricard,Le Castellet,France,68790,1,...,90.645,90.029,1.0,24.310,102.101604,53.0,1.000000,0.566667,10.0,10.0
161,997,2018,9,70,Austrian Grand Prix,Red Bull Ring,Spielberg,Austria,68810,1,...,63.577,63.149,1.0,21.245,69.929871,62.0,1.000000,0.666667,10.0,10.0
181,998,2018,10,9,British Grand Prix,Silverstone Circuit,Silverstone,UK,68830,1,...,86.256,85.892,1.0,28.982,101.000923,52.0,2.000000,0.766667,10.0,10.0


This feature "driver_circuit_performance_5" takes the performance metric of "finishing position" for a "driver-circuit" group (e.g hamilton_silverstone). This also works on the logic as before and looks as the average of the finishing position in the last 5 races (esentially telling us the recent form of the specific friver on a specific circuit). The lower the number i.e closer to 1 the better the form of the driver.

In [8]:
#    Pit stop time with the team
# first converting pit stop duration to numeric
df['pit_duration'] = pd.to_numeric(df['pit_duration'], errors='coerce')
# replacing 0 pit stop counts with NaN
df['pit_stop_count'] = df['pit_stop_count'].replace(0, np.nan)
df['avg_time_per_stop'] = df['pit_duration'] / df['pit_stop_count']
pit_groups = df.groupby([ 'constructorId'])['avg_time_per_stop']
# expanding average of past performance
df['constructor_avg_pit'] = pit_groups.transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)
# filling in the NaNs 
global_avg_pit_time = df['avg_time_per_stop'].mean()
df['constructor_avg_pit']= df['constructor_avg_pit'].fillna(global_avg_pit_time)

# review
print(df[['year', 'race_name', 'driver_name', 'constructorId', 'constructor_avg_pit']].head(10))


     year              race_name     driver_name  constructorId  \
1    2018  Australian Grand Prix  Lewis Hamilton              6   
39   2018     Bahrain Grand Prix  Lewis Hamilton            210   
41   2018     Chinese Grand Prix  Lewis Hamilton              6   
61   2018  Azerbaijan Grand Prix  Lewis Hamilton              6   
81   2018     Spanish Grand Prix  Lewis Hamilton              6   
101  2018      Monaco Grand Prix  Lewis Hamilton              6   
121  2018    Canadian Grand Prix  Lewis Hamilton              6   
141  2018      French Grand Prix  Lewis Hamilton              6   
161  2018    Austrian Grand Prix  Lewis Hamilton              6   
181  2018     British Grand Prix  Lewis Hamilton              6   

     constructor_avg_pit  
1              24.871523  
39             24.871523  
41             21.821000  
61             22.142500  
81             21.554000  
101            21.686750  
121            22.097200  
141            22.303500  
161            22.5

This feature "constructor_avg_pit" takes a look at the construstor effeciency, how fast or slow is the team with pit stop durations. We have created "avg_time_per_stop" that is (pit stop duration / pit stop count). Thois then is used as performance metric. The lower the time the better the team.

In [9]:

# We group by the team (constructorId) and the track (circuitId)
# We will use 'final_position' as our performance metric
constructor_circuit_groups = df.groupby(
    ['constructorId', 'circuitId'])['final_position']

# This creates a feature for the average finish of all drivers
# for that team at that circuit, using only past data.
df['constructor_circuit_avg_finish'] = constructor_circuit_groups.transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)

# Fill NaNs (for a team's first race at a circuit) with a neutral 10.0
df['constructor_circuit_avg_finish'] = df['constructor_circuit_avg_finish'].fillna(10.0)



For this feature "constructor_circuit_avg_finish" it makes a constructor-circuit group and averages the finishing position for the team on the specific circuit. For example where does ferrari on average finish at Monza. This will give us the impact of how teams perform on specific circuits, what team is the best at each circuit (with the lowest "constructor_circuit_avg_finish")

In [10]:
# Driver Circuit Experience
df['driver_circuit_experience'] = df.groupby(['driverId', 'circuitId']).cumcount()
# this creates a driver circuit group and counts how many times have the specific driver races on a certain track.The more amount of times the driver has raced the better his chances to finish better than someone who has not raced as many times

In [11]:
# Group by Year + Driver
driver_groups = df.groupby(['year', 'driverId'])['qualifying_position']

df['driver_qualifying_form'] = driver_groups.transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)
# Fill NaNs with 10.0 (neutral)
df['driver_qualifying_form'] = df['driver_qualifying_form'].fillna(10.0)

Driver qualifying form: average of qualifying position from the last 5 races, this will tell us is the driver has good qualifying position there are more chances for him to finish in a better place as well (as he started at a better position)

In [12]:
#calculating teams average finish for each race
df['team_race_avg_finish'] = df.groupby(['raceId', 'constructorId'])['final_position'].transform('mean')

#calculate the difference (Driver Finish - Team Avg)
# Negative value = Driver beat the team average (GOOD)
df['rel_teammate_performance'] = df['final_position'] - df['team_race_avg_finish']

df['driver_teammate_diff_form'] = df.groupby(['year', 'driverId'])['rel_teammate_performance'].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)
# Fill NaNs with 0.0 (performing exactly at team average)
df['driver_teammate_diff_form'] = df['driver_teammate_diff_form'].fillna(0.0)

This feature "driver teammate difference form" first takes the average of the "team" finishing postion and then subtracts the final position of the driver with the avg finishing position. It calculates it on a rolling basis for the previous 5 races. It can be used to seperate driver skill from the car itself, if the driver and his teammate finishing difference is more (negative in the case of how the formula is designed eg hamilton fininshed 2nd and his teammate finished 10th, the avg = 6 and the form= 2-6= -4). This can likely be associated with the driver skill (Current Form)

In [13]:

df['driver_teammate_diff_historic'] = df.groupby(['year', 'driverId'])['rel_teammate_performance'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)
# Fill NaNs with 0.0 (performing exactly at team average)
df['driver_teammate_diff_form'] = df['driver_teammate_diff_historic'].fillna(0.0)
# This is a smilar feature instead it captures the historic dominance of a driver over his teammate, unlike the first one where it was for the last five races

In [14]:

# Group by Year + Driver
driver_groups = df.groupby(['year', 'driverId'])['final_position']

# Calculate rolling Standard Deviation (std) of Final Position over the last 5 races
df['driver_consistency_std'] = driver_groups.transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).std()
)

# Impute NaNs (first few races require 2 data points for std dev) with 2.0 (neutral volatility)
df['driver_consistency_std'] = df['driver_consistency_std'].fillna(2.0)
# driver consistency, the lower std it has the more consistent it is

In [15]:
df.to_csv("master_dataset_final.csv", index=False)